In [1]:
import sys

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig, EmbedderModel, EmbedderModelConfig, VectorDBConnectionConfig
from src.knowledge_graph_model import KnowledgeGraphModel

from src.qa_pipeline.knowledge_retriever.cache import KeyValueStore, KeyValueStoreConfig

/home/dzigen/Desktop/PersonalAI/Personal-AI/pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri='bolt://localhost:7687', user='neo4j', pwd='password', db_name='diaasq2'),
    embeddings_db=EmbeddingsDatabaseConnection(EmbeddingsDatabaseConnectionConfig(
        embedder_config=EmbedderModelConfig(model_name_or_path='../../models/intfloat/multilingual-e5-small'),
        nodes_db_config=VectorDBConnectionConfig(
            '../../data/graph_structures/vectorized_nodes/v10/densedb', 'vectorized_nodes', is_exist=True, need_to_clear=False
        ),
        triplets_db_config=VectorDBConnectionConfig(
            '../../data/graph_structures/vectorized_triplets/v6/densedb', 'vectorized_triplets', is_exist=True, need_to_clear=False
        )
    ))
)

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [6]:
kg_model.embeddings_db.vectordbs['triplets'].collection.count()

72280

In [ ]:
@dataclass
class AStarMetricsConfig:
    h_metric_name: str = 'weight_with_short_path'
    d_metric_name: str = 'ip'

In [ ]:


@dataclass
class AStarGraphSearchConfig:
    metrics_config: AStarMetricsConfig = field(default_factory=lambda: AStarMetricsConfig())
    max_depth: int = 25
    accepted_node_types: List[str] = f'["{NodeType.object.value}","{NodeType.hyper.value}","{NodeType.episodic.value}"]'

class AStarMetrics:
    def __init__(self, kg_model: KnowledgeGraphModel, config: AStarMetricsConfig = AStarMetricsConfig(), cache: KeyValueStore = None):
        self.kg_model = kg_model
        self.cache = cache
        self.config = config
        self.metrics_map = {
            'l2': self.precomputed_dist,
            'ip': self.precomputed_dist,
            'constant': lambda v1, v2, U, parent: 1,
            'weight_with_short_path': self.weighted_short_path,
            'avg_weighted_with_short_path': self.avg_weighted_short_path,
        }

    def compute_d_metric(self, *args, **kwargs) -> float:
        return self.metrics_map[self.config.d_metric_name](*args, **kwargs)

    def compute_h_metric(self, *args, **kwargs) -> float:
        return self.metrics_map[self.config.h_metric_name](*args, **kwargs)

    def get_nodes_path(self, parent: Dict[str, str], end_node_id: str, spare_closest_node_id: str) -> List[str]:
        end_node_id = spare_closest_node_id if end_node_id not in parent else end_node_id
        
        path, end_flag, cur_n = [end_node_id], False, end_node_id
        while not end_flag:
            next_n = parent[cur_n]
            if next_n is None:
                end_flag = True
            else:
                path.append(next_n)
                cur_n = next_n

        return path

    def precomputed_dist(self, node1_id: str, node2_id: str, U: List[str], parent: Dict[str, str]) -> float:
        instances = self.config.nodes_db.read([node1_id, node2_id], includes=['embeddings'])
        dist = 1-np.dot(instances[0].embedding, instances[1].embedding)
        #print(dist, instances[0].id, instances[1].id)
        return dist

        #return self.nodes_dists['MATRIX'][self.nodes_dists['ID_TO_INDEX_MAP'][node1_id]][self.nodes_dists['ID_TO_INDEX_MAP'][node2_id]]
        
    def weighted_short_path(self, node1_id: str, node2_id: str, U: List[str], parent: Dict[str, str]) -> float:
        short_dist = self.nodes_short_paths['MATRIX'][self.nodes_short_paths['ID_TO_INDEX_MAP'][node1_id]][self.nodes_short_paths['ID_TO_INDEX_MAP'][node2_id]]
        w = self.precomputed_dist(node1_id, node2_id, U, parent)
        return short_dist * w

    # TODO
    def avg_weighted_short_path(self, node1_id: str, node2_id: str, U: List[str], parent: Dict[str, str]) -> float:
        nodes_path = self.get_nodes_path(parent, U, node1_id, None)
        acc_dist = 0
        for i in range(len(nodes_path)-1):
            acc_dist += self.precomputed_dist(nodes_path[i], nodes_path[i+1], U, parent)
        acc_dist += self.precomputed_dist(node1_id, node2_id, U, parent)

        short_dist = self.nodes_short_paths['MATRIX'][self.nodes_short_paths['ID_TO_INDEX_MAP'][node1_id]][self.nodes_short_paths['ID_TO_INDEX_MAP'][node2_id]]
        return np.mean(acc_dist) * short_dist 
